# QuizForge – Question Generation Demo

**Stages:** text extraction → cleaning → chunking → Seq2Seq LLM (Flan-T5) question generation → answer → TF-IDF keywords.

## 1. Install requirements

In [1]:
!pip install -q -r ../requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and project path

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so `import qgen` works from notebooks/

from qgen.extractor import extract_text, clean_text, build_chunks, select_chunks
from qgen.generator import QuestionGenerator, MODEL_OPTIONS
from qgen.pipeline import generate_quiz

E:\ai\envs\main\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load a textbook (PDF or DOCX)

In [3]:
BOOK_PATH = "D:\MIT-AoE\Academics\TY\DAA\Study Material\DAA_Unit_2\Genetic_Optimization_Topics.pdf"   ##

with open(BOOK_PATH, "rb") as f:
    raw_text = extract_text(BOOK_PATH, f.read())
print(len(raw_text.split()), "words extracted")
print(raw_text[:500])

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Om Channawar\AppData\Local\Temp\ipykernel_32988\198315389.py:1: SyntaxWarning: invalid escape sequence '\M'
  BOOK_PATH = "D:\MIT-AoE\Academics\TY\DAA\Study Material\DAA_Unit_2\Genetic_Optimization_Topics.pdf"   ##


5372 words extracted
Syllabus: 
Genetic Algorithm: Introduction to GA, Simple Genetic Algorithm, terminology and 
operators of GA (individual, gene, fitness, population, data structure, encoding, 
selection, crossover, mutation, convergence criteria). Reasons for working of GA and 
Schema theorem, GA optimization problems like TSP (Travelling salesman problem), 
Network design routing. Introduction to Ant Colony optimization (ACO) and Particle 
swarm optimization (PSO). 
 
Genetic Algorithm (GA) 
 Genetic Algorithm


## 4. Clean and chunk the text

In [4]:
text = clean_text(raw_text)
chunks = build_chunks(text)
print(len(chunks), "usable passages")
print("\nExample passage:\n", chunks[0])

33 usable passages

Example passage:
 Syllabus: Genetic Algorithm: Introduction to GA, Simple Genetic Algorithm, terminology and operators of GA (individual, gene, fitness, population, data structure, encoding, selection, crossover, mutation, convergence criteria). Reasons for working of GA and Schema theorem, GA optimization problems like TSP (Travelling salesman problem), Network design routing. Introduction to Ant Colony optimization (ACO) and Particle swarm optimization (PSO). Genetic Algorithm (GA)  Genetic Algorithm (GA) is a search-based optimization technique based on the principles of Genetics and Natural Selection.  It is frequently used to find optimal or near-optimal solutions to difficult problems which otherwise would take a lifetime to solve.  It is frequently used to solve optimization problems, in research, and in machine learning.  Terminology in Genetic Algorithms 1. Chromosome: A single solution to the problem. It is often represented as a string (e.g., binary, 

## 5. Load the Seq2Seq Transformer LLM

In [5]:
generator = QuestionGenerator("google/flan-t5-base")
print("Running on:", generator.device)

E:\ai\envs\main\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\ai\cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
E:\ai\envs\main\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to

Running on: cuda


## 6. Generate questions
Choose `difficulty` = `low` / `mid` / `high` and `NUM_QUESTIONS` = multiple of 5 (max 30).

In [10]:
DIFFICULTY = "mid"
NUM_QUESTIONS = 10

items = generate_quiz(text, generator, DIFFICULTY, NUM_QUESTIONS,
                      progress_callback=lambda p: print(f"\rprogress: {p:.0%}", end=""))
print()
for i, it in enumerate(items, 1):
    print(f"Q{i}. {it.question}")

progress: 100%
Q1. What is the basic unit of a chromosome?
Q2. How is crossover different from normal reproduction?
Q3. How can you select the best solution?
Q4. How can a Genetic Algorithm be applied to solve JSSP?
Q5. How can the algorithm stop when a maximum number of generations have passed?
Q6. How is crossover different from selection?
Q7. How does swarm intelligence differ from other types of intelligence?
Q8. How can ACO be applied to large problem spaces?
Q9. How is PSO different from other optimization algorithms?
Q10. How can BCO be used to optimize networks?


## 7. Answers and keywords

In [11]:
for i, it in enumerate(items, 1):
    print(f"Q{i}. {it.question}")
    print("   Keywords:", ", ".join(it.keywords))
    print("   Answer  :", it.answer, "\n")

Q1. What is the basic unit of a chromosome?
   Keywords: gene
   Answer  : Gene 

Q2. How is crossover different from normal reproduction?
   Keywords: helps, new, parts, parents, form, solutions
   Answer  : It helps combine the best parts of both parents to form new solutions. 

Q3. How can you select the best solution?
   Keywords: wheel, roulette, selection
   Answer  : Roulette Wheel Selection. 

Q4. How can a Genetic Algorithm be applied to solve JSSP?
   Keywords: makespan, evolving, crossover, potential, processes, selection
   Answer  : by evolving a population of potential solutions through selection, crossover, and mutation processes to minimize the makespan. 

Q5. How can the algorithm stop when a maximum number of generations have passed?
   Keywords: small, improvement, makespan, generations
   Answer  : when the improvement in the makespan becomes very small over several generations 

Q6. How is crossover different from selection?
   Keywords: new, combining, route, pare

## 8. (Optional) Compare difficulty levels on one passage
Shows how only the *prompt* changes the question style.

In [12]:
passage = chunks[0]
for level in ["low", "mid", "high"]:
    print(level.upper(), "->", generator.generate_question(passage, level))

LOW -> What is the basic unit of a chromosome?
MID -> What is the basic unit of a chromosome?
HIGH -> What is the basic unit of a chromosome?
